# Ordering

The same notebook as [`../ordering.ipynb`](../ordering.ipynb), written against
[`ib_async`](https://github.com/ib-api-reloaded/ib_async) instead of the TWS API
shape. The library is unmodified and installed as usual; `ibx.ib_async.attach`
replaces the one layer of it that expects a socket to a gateway.

Placing an order, watching it, and withdrawing it. Paper account.

## Connecting

`IB.connect` was written for a gateway, so it takes a host, a port and a client
id. Here it takes none of them: the credentials go to `attach`, and there is no
local process to reach.

`ib.sleep()` rather than `time.sleep()` throughout. The library's loop runs on
this thread, and a plain sleep stops it — every stream then reads as dead.

In [ ]:
import os
from dotenv import load_dotenv
from ib_async import IB, util
import ibx.ib_async

util.startLoop()
load_dotenv()

ib = ibx.ib_async.attach(
    IB(),
    username=os.environ["IB_USERNAME"],
    password=os.environ["IB_PASSWORD"],
    paper=True,
)
ib.connect()          # names no host: there is no gateway to name

print(f"connected: {ib.isConnected()}")
print(f"accounts:  {ib.managedAccounts()}")

## A price to work from

An order priced far from the market will not fill, which is what makes it
safe to place and withdraw.

In [ ]:
from ib_async import Stock

aapl = Stock("AAPL", "SMART", "USD")
ib.qualifyContracts(aapl)

ticker = ib.reqMktData(aapl)
ib.sleep(3)
print(f"bid {ticker.bid}  ask {ticker.ask}  last {ticker.last}")

## A limit order that will not fill

`placeOrder` hands back a `Trade`, which is amended as the venue reports on
it. Its log is the whole history of the order.

In [ ]:
from ib_async import LimitOrder

reference = ticker.last or ticker.close or 100.0
away = round(reference * 0.5, 2)

trade = ib.placeOrder(aapl, LimitOrder("BUY", 1, away))
ib.sleep(3)

print(f"status: {trade.orderStatus.status}")
for entry in trade.log:
    print(f"  {entry.time}  {entry.status}  {entry.message}")

## Moving it

A modification is the same order under the same id, at a new price.

In [ ]:
trade.order.lmtPrice = round(away * 0.98, 2)
ib.placeOrder(aapl, trade.order)
ib.sleep(3)
print(f"{trade.orderStatus.status} at {trade.order.lmtPrice}")

## Withdrawing it

In [ ]:
ib.cancelOrder(trade.order)
ib.sleep(3)
print(f"status: {trade.orderStatus.status}")

## What is still working

In [ ]:
open_trades = ib.openTrades()
print(f"{len(open_trades)} open")
for t in open_trades:
    o = t.order
    print(f"  {o.action} {o.totalQuantity} {t.contract.symbol} @ {o.lmtPrice}  {t.orderStatus.status}")

## What was previewed rather than sent

`whatIfOrder` asks the venue what an order would cost the account, and sends
nothing.

In [ ]:
preview = ib.whatIfOrder(aapl, LimitOrder("BUY", 100, away))
print(f"initial margin  {preview.initMarginChange}")
print(f"maint margin    {preview.maintMarginChange}")
print(f"commission      {preview.commission}")

In [ ]:
ib.cancelMktData(aapl)
ib.disconnect()